In [1]:
# S-00  Run configuration
# Ensemble submission: small_datasets model for retinamnist + breastmnist,
#                      general model for all other 9 datasets.
#
# HOW TO USE:
#   Set checkpoint paths to the best_model.pth files from each training run.
#   Run all cells top to bottom.
#   Do not modify any other cell.

from pathlib import Path

# Auto-detect environment: Kaggle vs local
_on_kaggle  = Path("/kaggle/working").exists()
WORKING_DIR = Path("/kaggle/working") if _on_kaggle else Path("..").resolve()

GENERAL_CKPT_PATH = Path("/kaggle/input/models/ionutcalin645/tensor-reloaded-mnist/pytorch/default/2/best_common_model.pth")
SMALL_CKPT_PATH   = Path("/kaggle/input/models/ionutcalin645/tensor-reloaded-mnist/pytorch/default/2/best_small_datasets_model.pth")

# Fallback backbone names — the checkpoint stores the actual name used during training
GENERAL_BACKBONE  = "convnext_tiny.in12k_ft_in1k"
SMALL_BACKBONE    = "convnext_tiny.in12k_ft_in1k"

IMAGE_SIZE        = 28
BATCH_SIZE        = 256
# num_workers > 0 can hang on Windows/WSL due to multiprocessing — use 0 locally
NUM_WORKERS       = 2 if _on_kaggle else 0
TTA_ENABLED       = True
SEED              = 42

# Must match DATASET_EMBED_DIM used when training the small-datasets model
DATASET_EMBED_DIM = 32

print("Environment:", "Kaggle" if _on_kaggle else "local")
print("Working dir: ", WORKING_DIR)
print("General checkpoint:", GENERAL_CKPT_PATH)
print("Small checkpoint:  ", SMALL_CKPT_PATH)
print("TTA:", TTA_ENABLED)

Environment: Kaggle
Working dir:  /kaggle/working
General checkpoint: /kaggle/input/models/ionutcalin645/tensor-reloaded-mnist/pytorch/default/2/best_common_model.pth
Small checkpoint:   /kaggle/input/models/ionutcalin645/tensor-reloaded-mnist/pytorch/default/2/best_small_datasets_model.pth
TTA: True


In [2]:
# S-01  Install dependencies
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "timm", "scikit-learn", "scipy", "pandas", "tqdm"])
print("Dependencies installed")

Dependencies installed


In [3]:
# S-02  Imports, seeds, output directories
import random
from datetime import datetime, UTC
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import timm
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

SEED = 69

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Auto-detect data root — covers: project root, notebooks/ subdir, Kaggle mounts
_data_candidates = [
    Path("data"),
    Path("../data"),
    Path("/kaggle/input/tensor-reloaded-multi-task-med-mnist/data"),
    Path("/kaggle/input/competitions/tensor-reloaded-multi-task-med-mnist/data"),
]
DATA_ROOT = next((p for p in _data_candidates if (p / "pathmnist.npz").exists()), None)
assert DATA_ROOT is not None, (
    "Data not found. Run: list(Path('.').rglob('*.npz')) to locate the files."
)

SUBMISSION_DIR = WORKING_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("device:", DEVICE)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  {props.total_memory/1e9:.1f} GB")
print("data root:", DATA_ROOT)
print("submissions:", SUBMISSION_DIR)

torch: 2.10.0+cu128
device: cuda
GPU: Tesla T4  15.6 GB
data root: /kaggle/input/competitions/tensor-reloaded-multi-task-med-mnist/data
submissions: /kaggle/working/submissions


In [4]:
# S-03  Dataset constants and data loading

ALL_DATASETS = [
    "pathmnist", "dermamnist", "octmnist", "pneumoniamnist",
    "retinamnist", "breastmnist", "bloodmnist", "tissuemnist",
    "organamnist", "organcmnist", "organsmnist",
]

# Datasets covered by the small_datasets model (must match DATASETS in small_datasets_run.ipynb)
SMALL_DATASETS = ["retinamnist", "breastmnist"]

DATASET_META = {
    "pathmnist":     {"n_classes": 9,  "n_channels": 3},
    "dermamnist":    {"n_classes": 7,  "n_channels": 3},
    "octmnist":      {"n_classes": 4,  "n_channels": 1},
    "pneumoniamnist":{"n_classes": 2,  "n_channels": 1},
    "retinamnist":   {"n_classes": 5,  "n_channels": 3},
    "breastmnist":   {"n_classes": 2,  "n_channels": 1},
    "bloodmnist":    {"n_classes": 8,  "n_channels": 3},
    "tissuemnist":   {"n_classes": 8,  "n_channels": 1},
    "organamnist":   {"n_classes": 11, "n_channels": 1},
    "organcmnist":   {"n_classes": 11, "n_channels": 1},
    "organsmnist":   {"n_classes": 11, "n_channels": 1},
}

NO_VFLIP = {"organamnist", "organcmnist", "organsmnist", "octmnist"}

EXPECTED_TEST_SIZES = {
    "pathmnist": 7180, "dermamnist": 2005, "octmnist": 1000,
    "pneumoniamnist": 624, "retinamnist": 400, "breastmnist": 156,
    "bloodmnist": 3421, "tissuemnist": 47280, "organamnist": 17778,
    "organcmnist": 8268, "organsmnist": 8829,
}

# Load all 11 datasets into RAM
# np.load() is lazy (keeps zip open). Calling .copy() forces eager load and
# prevents file-handle corruption when num_workers > 0.
print(f"{'Dataset':<18}  {'Test':>6}  {'Model'}")
print("-" * 44)
all_data = {}
for ds in ALL_DATASETS:
    path = DATA_ROOT / f"{ds}.npz"
    assert path.exists(), f"MISSING: {path}"
    with np.load(path) as _f:
        all_data[ds] = {k: _f[k].copy() for k in _f.files}
    n_te  = all_data[ds]["test_images"].shape[0]
    which = "small_datasets" if ds in SMALL_DATASETS else "general"
    print(f"{ds:<18}  {n_te:>6,}  {which}")
print("-" * 44)
total = sum(all_data[ds]["test_images"].shape[0] for ds in ALL_DATASETS)
print(f"Total test images: {total:,}")
assert total == 96941, f"Expected 96941, got {total}"

Dataset               Test  Model
--------------------------------------------
pathmnist            7,180  general
dermamnist           2,005  general
octmnist             1,000  general
pneumoniamnist         624  general
retinamnist            400  small_datasets
breastmnist            156  small_datasets
bloodmnist           3,421  general
tissuemnist         47,280  general
organamnist         17,778  general
organcmnist          8,268  general
organsmnist          8,829  general
--------------------------------------------
Total test images: 96,941


In [5]:
# S-04  Shared utilities: dataset class and val transform

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class MedDataset(Dataset):
    """Inference-only dataset — returns images only (no labels)."""
    def __init__(self, images, transform=None):
        self.images    = images
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        if img.ndim == 2:                              # grayscale (H, W) → (H, W, 3)
            img = np.stack([img, img, img], axis=2)
        from PIL import Image as PILImage
        img = PILImage.fromarray(img.astype(np.uint8), "RGB")
        if self.transform:
            img = self.transform(img)
        return img


def get_val_transform(image_size):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


print("Utilities defined.")

Utilities defined.


In [6]:
# S-05  Model definitions
#
# GeneralMultiTaskModel   — matches common_run.ipynb:
#   heads for all 11 datasets, no dataset embedding.
#
# SmallDatasetMultiTaskModel — matches small_datasets_run.ipynb:
#   heads for [retinamnist, breastmnist] only, learnable dataset embedding
#   concatenated to backbone features before each head.

MAX_CLASSES = max(m["n_classes"] for m in DATASET_META.values())  # 11


class GeneralMultiTaskModel(nn.Module):
    def __init__(self, backbone_name):
        super().__init__()
        self.task_n_classes = {ds: DATASET_META[ds]["n_classes"] for ds in ALL_DATASETS}

        self.backbone = timm.create_model(
            backbone_name, pretrained=False, num_classes=0, drop_path_rate=0.1
        )
        # Stem modification for 28×28 input: replace stride-4 conv with stride-1
        in_ch  = self.backbone.stem[0].in_channels
        out_ch = self.backbone.stem[0].out_channels
        self.backbone.stem[0] = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1)

        feat_dim = self.backbone.num_features  # 768 for convnext_tiny
        self.heads = nn.ModuleDict({
            ds: nn.Sequential(
                nn.LayerNorm(feat_dim),
                nn.Linear(feat_dim, feat_dim // 2),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(feat_dim // 2, self.task_n_classes[ds]),
            )
            for ds in ALL_DATASETS
        })


class SmallDatasetMultiTaskModel(nn.Module):
    def __init__(self, backbone_name, embed_dim=32):
        super().__init__()
        self.datasets       = SMALL_DATASETS
        self.task_n_classes = {ds: DATASET_META[ds]["n_classes"] for ds in self.datasets}
        self.embed_dim      = embed_dim

        self.backbone = timm.create_model(
            backbone_name, pretrained=False, num_classes=0, drop_path_rate=0.1
        )
        in_ch  = self.backbone.stem[0].in_channels
        out_ch = self.backbone.stem[0].out_channels
        self.backbone.stem[0] = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1)

        feat_dim = self.backbone.num_features
        if embed_dim > 0:
            self.ds_embedding = nn.Embedding(len(self.datasets), embed_dim)
            head_in_dim = feat_dim + embed_dim
        else:
            self.ds_embedding = None
            head_in_dim = feat_dim

        self.heads = nn.ModuleDict({
            ds: nn.Sequential(
                nn.LayerNorm(head_in_dim),
                nn.Linear(head_in_dim, head_in_dim // 2),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(head_in_dim // 2, self.task_n_classes[ds]),
            )
            for ds in self.datasets
        })

    def get_features_with_embed(self, images, ds_name):
        feats = self.backbone(images)
        if self.embed_dim > 0:
            ds_idx = torch.tensor(
                self.datasets.index(ds_name), device=images.device
            ).expand(len(images))
            feats = torch.cat([feats, self.ds_embedding(ds_idx)], dim=1)
        return feats


print(f"Model classes defined. MAX_CLASSES={MAX_CLASSES}")

Model classes defined. MAX_CLASSES=11


In [7]:
# S-06  Load both checkpoints

def resolve_backbone(name):
    """Return name if available in this timm version, else a known fallback."""
    available = timm.list_models()
    if name in available:
        return name
    for fb in ["convnext_tiny.fb_in22k_ft_in1k", "convnext_tiny"]:
        if fb in available:
            print(f"WARNING: {name} not found — using fallback: {fb}")
            return fb
    raise ValueError(
        f"{name} not found in timm {timm.__version__}.\n"
        f"Available convnext_tiny variants: "
        f"{[m for m in available if 'convnext_tiny' in m]}"
    )


assert GENERAL_CKPT_PATH.exists(), f"General checkpoint not found: {GENERAL_CKPT_PATH}"
assert SMALL_CKPT_PATH.exists(),   f"Small checkpoint not found: {SMALL_CKPT_PATH}"

# ── General model ─────────────────────────────────────────────────────────────
general_ckpt     = torch.load(GENERAL_CKPT_PATH, map_location=DEVICE)
general_backbone = resolve_backbone(general_ckpt.get("backbone", GENERAL_BACKBONE))
general_model    = GeneralMultiTaskModel(general_backbone).to(DEVICE)
general_model.load_state_dict(general_ckpt["model_state_dict"])
general_model.eval()
print(f"General model loaded  — epoch {general_ckpt['epoch']}, "
      f"val_hm={general_ckpt['val_hm']:.4f}")

# ── Small-datasets model ──────────────────────────────────────────────────────
small_ckpt     = torch.load(SMALL_CKPT_PATH, map_location=DEVICE)
small_backbone = resolve_backbone(small_ckpt.get("backbone", SMALL_BACKBONE))
small_model    = SmallDatasetMultiTaskModel(small_backbone, embed_dim=DATASET_EMBED_DIM).to(DEVICE)
small_model.load_state_dict(small_ckpt["model_state_dict"])
small_model.eval()
print(f"Small model loaded    — epoch {small_ckpt['epoch']}, "
      f"val_hm={small_ckpt['val_hm']:.4f}")

print("\nDataset → model mapping:")
for ds in ALL_DATASETS:
    which = "small_datasets" if ds in SMALL_DATASETS else "general"
    print(f"  {ds:<18} → {which}")

General model loaded  — epoch 20, val_hm=0.7812
Small model loaded    — epoch 35, val_hm=0.6318

Dataset → model mapping:
  pathmnist          → general
  dermamnist         → general
  octmnist           → general
  pneumoniamnist     → general
  retinamnist        → small_datasets
  breastmnist        → small_datasets
  bloodmnist         → general
  tissuemnist        → general
  organamnist        → general
  organcmnist        → general
  organsmnist        → general


In [8]:
# S-07  Generate predictions
# General model: plain backbone → head.
# Small model:   backbone + dataset embedding → head (via get_features_with_embed).
# TTA: horizontal flip always safe; vertical flip disabled for NO_VFLIP datasets.

val_tf    = get_val_transform(IMAGE_SIZE)
_pin_mem  = torch.cuda.is_available()


def _make_loader(ds_name):
    ds = MedDataset(all_data[ds_name]["test_images"], transform=val_tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=_pin_mem)


def get_general_probs(ds_name, tta=False):
    all_probs = []
    with torch.no_grad():
        for images in _make_loader(ds_name):
            images = images.to(DEVICE)
            logits_list = [general_model.heads[ds_name](general_model.backbone(images))]
            if tta:
                hflip = torch.flip(images, dims=[3])
                logits_list.append(general_model.heads[ds_name](general_model.backbone(hflip)))
                if ds_name not in NO_VFLIP:
                    vflip = torch.flip(images, dims=[2])
                    logits_list.append(general_model.heads[ds_name](general_model.backbone(vflip)))
            probs = torch.stack([l.softmax(dim=1) for l in logits_list]).mean(0)
            all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)


def get_small_probs(ds_name, tta=False):
    all_probs = []
    with torch.no_grad():
        for images in _make_loader(ds_name):
            images = images.to(DEVICE)
            feats  = small_model.get_features_with_embed(images, ds_name)
            logits_list = [small_model.heads[ds_name](feats)]
            if tta:
                hflip       = torch.flip(images, dims=[3])
                feats_hflip = small_model.get_features_with_embed(hflip, ds_name)
                logits_list.append(small_model.heads[ds_name](feats_hflip))
                if ds_name not in NO_VFLIP:
                    vflip       = torch.flip(images, dims=[2])
                    feats_vflip = small_model.get_features_with_embed(vflip, ds_name)
                    logits_list.append(small_model.heads[ds_name](feats_vflip))
            probs = torch.stack([l.softmax(dim=1) for l in logits_list]).mean(0)
            all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)


test_predictions = {}
for ds_name in tqdm(ALL_DATASETS, desc="Predicting"):
    if ds_name in SMALL_DATASETS:
        probs = get_small_probs(ds_name, tta=TTA_ENABLED)
    else:
        probs = get_general_probs(ds_name, tta=TTA_ENABLED)
    test_predictions[ds_name] = probs.argmax(axis=1)
    which = "small" if ds_name in SMALL_DATASETS else "general"
    print(f"{ds_name:<18} [{which}]  {len(probs):>6,} preds  "
          f"classes: {sorted(set(test_predictions[ds_name].tolist()))}")

Predicting:   0%|          | 0/11 [00:00<?, ?it/s]

/tmp/ipykernel_23/2001982078.py:21: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), "RGB")
/tmp/ipykernel_23/2001982078.py:21: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), "RGB")


pathmnist          [general]   7,180 preds  classes: [0, 1, 2, 3, 4, 5, 6, 7, 8]


/tmp/ipykernel_23/2001982078.py:21: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), "RGB")
/tmp/ipykernel_23/2001982078.py:21: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), "RGB")


dermamnist         [general]   2,005 preds  classes: [0, 1, 2, 3, 4, 5, 6]


...

organamnist        [general]  17,778 preds  classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


/tmp/ipykernel_23/2001982078.py:21: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), "RGB")
/tmp/ipykernel_23/2001982078.py:21: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), "RGB")


organcmnist        [general]   8,268 preds  classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


/tmp/ipykernel_23/2001982078.py:21: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), "RGB")
/tmp/ipykernel_23/2001982078.py:21: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), "RGB")


organsmnist        [general]   8,829 preds  classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [9]:
# S-08  Build and validate submission CSV
# Competition requires EXACTLY this column order: id, id_image_in_task, task_name, label

timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")

rows, global_id = [], 0
for ds_name in ALL_DATASETS:
    preds = test_predictions[ds_name]
    for img_id, label in enumerate(preds):
        rows.append({
            "id":               global_id,
            "id_image_in_task": img_id,
            "task_name":        ds_name,
            "label":            int(label),
        })
        global_id += 1

submission_df = pd.DataFrame(rows)

assert list(submission_df.columns) == ["id", "id_image_in_task", "task_name", "label"], \
    f"Wrong column order: {list(submission_df.columns)}"
assert len(submission_df) == 96941, \
    f"Wrong row count: {len(submission_df)} (expected 96941)"
assert not submission_df["label"].isna().any(), "NaN labels found"
assert submission_df["id"].is_monotonic_increasing, "id not monotonically increasing"
for ds, expected in EXPECTED_TEST_SIZES.items():
    actual = len(submission_df[submission_df["task_name"] == ds])
    assert actual == expected, f"{ds}: expected {expected} rows, got {actual}"

submission_path = SUBMISSION_DIR / f"submission_ensemble_{timestamp}.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved: {submission_path}")
print(f"Total rows: {len(submission_df):,}")
print(f"Columns: {list(submission_df.columns)}")
display(submission_df.head(8))

Submission saved: /kaggle/working/submissions/submission_ensemble_20260527_102653.csv
Total rows: 96,941
Columns: ['id', 'id_image_in_task', 'task_name', 'label']


,id,id_image_in_task,task_name,label
0,0,0,pathmnist,8
1,1,1,pathmnist,4
2,2,2,pathmnist,4
3,3,3,pathmnist,3
4,4,4,pathmnist,4
5,5,5,pathmnist,4
6,6,6,pathmnist,8
7,7,7,pathmnist,0
